# Lecture 7: Matrix decompositions review. How to compute the QR decomposition and the Schur decomposition

In [ ]:
# Setup. Requirements: pip install numpy scipy matplotlib
import numpy as np
import scipy.linalg as sla
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({'figure.figsize': (7, 4), 'figure.dpi': 110})
rng = np.random.default_rng(0)


def orth_error(Q: np.ndarray) -> float:
    # loss of orthogonality ||Q^T Q - I||_F
    return np.linalg.norm(Q.T @ Q - np.eye(Q.shape[1]))


## Recap of the previous lecture
- Eigenvectors and eigenvalues
- Characteristic polynomial and why it is a bad idea
- Power method to find the leading (maximum absolute value) eigenvalue and eigenvector
- Gershgorin theorem
- Schur theorem: $A = U T U^*$
- Normal matrices: $A^* A = A A^*$
- Advanced topic: pseudospectrum

## Today's lecture

- Today we will talk about **matrix factorizations** as a general tool.

- Basic **matrix factorizations** in numerical linear algebra:

    - LU decomposition and Gaussian elimination &mdash; already covered
    - QR decomposition and the Gram&ndash;Schmidt algorithm
    - Schur decomposition and the QR algorithm
    - Methods for computing the SVD in the second part

- We already introduced the QR decomposition some time ago, but now we are going to discuss it in more detail.
- New today: the QR algorithm is the **power method in disguise**.

## General concept of matrix factorization

- In numerical linear algebra we need to solve different tasks, for example:

    - Solve linear systems $Ax = f$
    - Compute eigenvalues / eigenvectors
    - Compute singular values / singular vectors
    - Compute inverses, sometimes even determinants
    - Compute **matrix functions** like $\exp(A), \cos(A)$ (these are not elementwise functions)

- In order to do this, we represent the matrix as a sum and/or product of matrices with **simpler structure**, such that we can solve the mentioned tasks faster and/or in a more stable way.

- What is a **simpler structure**?

## What is a simpler structure

- We already encountered several classes of matrices **with structure**.

- For dense matrices the most important classes are

    - **unitary matrices**: $Q^{-1} = Q^*$ and $\kappa_2(Q) = 1$, so they do not amplify errors;
    - **upper/lower triangular matrices**: linear systems are solved in $\mathcal{O}(n^2)$ operations, the eigenvalues are on the diagonal;
    - **diagonal matrices**: everything is done elementwise.

## Other classes of structured matrices

- For **sparse matrices** the sparsity constraints are often included in the factorizations.
- For **Toeplitz matrices** an important class of matrices is the class of matrices with **low displacement rank**, which is based on low-rank matrices.
- The class of **low-rank matrices** and **block low-rank matrices** appears in many applications.

## Plan
The plan for today's lecture is to discuss the decompositions one by one and point out:
- How to compute a particular decomposition
- When the decomposition exists
- What is done in **real life** (LAPACK).

## Decompositions we want to discuss today
- LU factorization & Cholesky factorization &mdash; quick reminder, already done.
- QR decomposition: the Gram&ndash;Schmidt algorithm and Householder reflections.
- Schur decomposition: the QR algorithm and its connection to the power method.
- One slide about the SVD (more details in the second part of today's lecture).

## PLU decomposition

- Any nonsingular matrix can be factored as

$$ A = P L U, $$

where $P$ is a permutation matrix, $L$ is a **lower triangular matrix** (with unit diagonal), and $U$ is **upper triangular**.

- The **main goal** of the LU decomposition is to solve linear systems, because

$$ A^{-1} f = (P L U)^{-1} f = U^{-1} L^{-1} P^\top f, $$

and this reduces to the solution of two linear systems

$$ L y = P^\top f,  \quad U x = y $$

with lower and upper triangular matrices respectively.

**Q**: what is the complexity of solving these linear systems?

<details><summary><b>Answer</b></summary>

$\mathcal{O}(n^2)$: forward and back substitution take about $n^2$ operations each, while the factorization itself costs $\frac23 n^3$. So we factorize once and then solve cheaply with many right-hand sides.

</details>

## Positive definite matrices and Cholesky decomposition, reminder

If the matrix is Hermitian positive definite, i.e.

$$ A = A^*, \quad (Ax, x) > 0, \quad x \ne 0, $$

then it can be factored as

$$ A = R^* R, $$

where $R$ is **upper** triangular with a positive diagonal (equivalently, $A = LL^*$ with the lower triangular $L = R^*$).

We will need this for the QR decomposition.

## QR decomposition

- The next decomposition: the **QR** decomposition.
- Again, from the name it is clear that a matrix is represented as a product

$$
    A = Q R,
$$

where $Q$ has **orthonormal columns** ($Q^* Q = I$) and $R$ is **upper triangular**.

- The matrix sizes for $A \in \mathbb{C}^{n\times m}$, $n \geq m$:
    - **thin (economy) QR**: $Q$ is $n \times m$, $R$ is $m \times m$;
    - **full QR**: $Q$ is $n \times n$ unitary, $R$ is $n \times m$ with zero last $n - m$ rows.

  See [poster](decompositions.pdf) for a visualization of the QR decomposition.

- The QR decomposition is defined for **any rectangular matrix**.

## QR decomposition: applications

This decomposition plays a crucial role in many problems:
- Computing orthogonal bases in a linear space
- Used in the preprocessing step for the SVD
- The QR algorithm for the computation of eigenvectors and eigenvalues ([one of the 10 most important algorithms of the 20th century](https://archive.siam.org/pdf/news/637.pdf)) is based on the QR decomposition
- Solving overdetermined systems of linear equations (linear least-squares problem)

## QR decomposition and least squares, reminder

- Suppose we need to solve

$$ \Vert A x - b \Vert_2 \rightarrow \min_x, $$

where $A$ is $n \times m$, $n \geq m$.

- Then we factorize (thin QR)

$$ A = Q R, $$

and use the formula for the pseudoinverse in the case of a **full column** rank matrix $A$:

$$ x = A^{\dagger}b = (A^*A)^{-1}A^*b = ((QR)^*(QR))^{-1}(QR)^*b = (R^*Q^*QR)^{-1}R^*Q^*b = R^{-1}Q^*b, $$

thus $x$ can be recovered from

$$R x = Q^*b.$$

- Note that this is a square system of linear equations with an **upper** triangular matrix. What is the complexity of solving this system?
- The matrix $A^*A$ appears here only on paper: in computations it is never formed (we will see why in a minute).

<details><summary><b>Answer</b></summary>

$\mathcal{O}(m^2)$ (back substitution). The total cost is dominated by the QR decomposition, $\mathcal{O}(nm^2)$.

</details>

## Existence of QR decomposition
**Theorem.**
Every rectangular $n \times m$ matrix has a QR decomposition.

There are several ways to prove it and compute it:

- Theoretical: using the Gram matrices and the Cholesky factorization
- Geometrical: using the Gram&ndash;Schmidt orthogonalization
- Practical: using Householder/Givens transformations

### Proof using Cholesky decomposition

If we have a representation of the form

$$A = QR,$$

then $A^* A = ( Q R)^* (QR)  = R^* (Q^* Q) R = R^* R$. The matrix $A^* A$ is called the **Gram matrix**; its elements are scalar products of the columns of $A$. So $R$ must be the Cholesky factor of the Gram matrix.

### Proof using Cholesky decomposition (full column rank)

- Assume that $A$ has **full column rank**. Then it is simple to show that $A^* A$ is positive definite:

$$ (A^* A y, y) = (Ay, Ay) = \Vert Ay \Vert^2  > 0, \quad y\not = 0. $$

- Therefore, the Cholesky decomposition $A^* A = R^* R$ with an upper triangular nonsingular $R$ always exists.

- Then the matrix $Q = A R^{-1}$ has orthonormal columns:

$$ (A R^{-1})^* (AR^{-1})= R^{-*} A^* A R^{-1} = R^{-*} R^* R R^{-1} = I. $$

### Proof using Cholesky decomposition (rank-deficient case)

- When an $n \times m$ matrix does not have full column rank, it is said to be **rank-deficient**.

- The QR decomposition, however, also exists.

- For any **rank-deficient matrix** there is a sequence of full column rank matrices $B_j \rightarrow A$ (**why?**).

- Each $B_j$ can be decomposed as $B_j = Q_j R_j$.

- The set of $n\times m$ matrices with orthonormal columns is compact (**why?**), thus there exists a converging subsequence $Q_{j_l} \rightarrow Q$, and still $Q^*Q = I$.

- Then $R_{j_l} = Q_{j_l}^* B_{j_l} \rightarrow Q^* A =: R$, which is upper triangular as a limit of upper triangular matrices, and $A = \lim B_{j_l} = \lim Q_{j_l} R_{j_l} = QR$.

<details><summary><b>Answers</b></summary>

- Take the SVD $A = U\Sigma V^*$ and replace the zero singular values by $1/j$: the resulting $B_j$ has full column rank and $\|B_j - A\|_2 \le 1/j$.
- The set is bounded (every column has unit norm) and closed (it is given by the equation $Q^*Q = I$), hence compact in finite dimension.
- Now $R$ may have zeros on the diagonal, and the decomposition is not unique.

</details>

## Stability of QR decomposition via Cholesky decomposition

- So, the simplest way to compute the QR decomposition is

$$A^* A = R^* R,$$

and

$$Q = A R^{-1}.$$

- It is a **bad idea** for numerical stability. Let us do a demo (for a submatrix of the Hilbert-like matrix $a_{ji} = 1/(i + j + 0.5)$).

In [ ]:
n, r = 10, 7
i, j = np.arange(r), np.arange(n)
a = 1.0 / (i[None, :] + j[:, None] + 0.5)          # a[j, i] = 1 / (i + j + 0.5), size 10 x 7
print(f'cond(a) = {np.linalg.cond(a):.1e}')

q, R = np.linalg.qr(a)
print(f'built-in QR:  ||Q^T Q - I|| = {orth_error(q):.1e}')

L = np.linalg.cholesky(a.T @ a)                     # a^T a = L L^T, i.e. R = L^T
q1 = sla.solve_triangular(L, a.T, lower=True).T     # q1 = a L^{-T} = a R^{-1}
print(f'via Cholesky: ||Q^T Q - I|| = {orth_error(q1):.1e}')


The built-in QR (Householder reflections, see below) keeps orthogonality at the level of machine precision. Via the Gram matrix the loss of orthogonality is of order $\kappa(A)^2 \varepsilon_{\rm mach}$: forming $A^*A$ squares the condition number. For $\kappa(A) \gtrsim 10^8$ the Cholesky factorization of the computed Gram matrix may even fail.

## Second way: Gram&ndash;Schmidt orthogonalization

- The QR decomposition is a mathematical way of writing down the Gram&ndash;Schmidt orthogonalization process.
- Given linearly independent vectors $a_1, \ldots, a_m$, we want to find an **orthonormal** basis $q_1, \ldots, q_m$ such that $\operatorname{span}(q_1, \dots, q_k) = \operatorname{span}(a_1, \dots, a_k)$ for every $k$.

**Gram&ndash;Schmidt:**
1. $q_1 := a_1/\Vert a_1 \Vert$
2. $q_2 := a_2 - (a_2, q_1) q_1, \quad q_2 := q_2/\Vert q_2 \Vert$
3. $q_3 := a_3 - (a_3, q_1) q_1 - (a_3, q_2) q_2, \quad q_3 := q_3/\Vert q_3 \Vert$
4. And so on.

Why this is a QR decomposition: at step $k$ we obtain

$$a_k = r_{1k} q_1 + \dots + r_{kk} q_k, \qquad r_{ik} = (a_k, q_i), \quad r_{kk} = \Big\Vert a_k - \sum_{i<k} r_{ik} q_i \Big\Vert,$$

so the $k$-th column of $A$ uses only $q_1, \dots, q_k$, i.e. $A = QR$ with an upper triangular $R$.

## Modified Gram&ndash;Schmidt
- Gram&ndash;Schmidt can be very unstable: the produced vectors are not orthogonal, especially if $a_k - \sum_{i<k} (a_k, q_i) q_i$ has a small norm (strong cancellation). This is called **loss of orthogonality**.

- There is a remedy, called the **modified Gram&ndash;Schmidt** method. Instead of doing

$$q_k := a_k - (a_k, q_1) q_1 - \ldots - (a_k, q_{k-1}) q_{k-1}$$

we do it step by step. First we set $q_k := a_k$ and orthogonalize sequentially:

$$
   q_k := q_k - (q_k, q_1)q_1, \quad q_k := q_{k} - (q_k,q_2)q_2, \ldots
$$

- In exact arithmetic, it is the same. In floating point arithmetic it is absolutely different!

- The complexity of both versions is $\mathcal{O}(nm^2)$ operations (about $2nm^2$).

## Loss of orthogonality in one step

Take a vector $b$ that is almost parallel to a unit vector $a$, and orthogonalize it against $a$ once, and then once more.

In [ ]:
n = 100
a = rng.random(n)
b = a + 1e-9 * rng.standard_normal(n)       # almost parallel to a
a /= np.linalg.norm(a)

c = b - (b @ a) * a                         # one Gram-Schmidt step
c /= np.linalg.norm(c)
print(f'after one pass:   (c, a) = {c @ a: .1e}')

c = c - (c @ a) * a                         # the same step once more
c /= np.linalg.norm(c)
print(f'after two passes: (c, a) = {c @ a: .1e}')


The subtraction $b - (b, a)\,a$ cancels about 9 leading digits, and the rounding error of order $10^{-16}\|b\|$ becomes relatively large in the small remainder: $(c, a)$ is far from $0$. Repeating the orthogonalization (**reorthogonalization**) restores orthogonality to machine precision; a classical rule says that &ldquo;twice is enough&rdquo;.

## QR decomposition: the (almost) practical way

- If $A = QR$ (full QR), then

$$ R = Q^* A, $$

and we need to find a unitary matrix $Q$ that brings a matrix into upper triangular form.
- For simplicity, we look for an $n \times n$ matrix $Q$ such that

$$ Q^* A = \begin{bmatrix} * & * & *  \\ 0 & * & * \\ 0 & 0 & * \\ & 0_{(n-m) \times m} \end{bmatrix}. $$

- We will do it column by column with **Householder reflections**

$$H = I - 2uu^*, \qquad \|u\|_2 = 1.$$

$H$ is the reflection across the hyperplane $u^\perp$; it is Hermitian and unitary, $H = H^* = H^{-1}$.
- For a real vector $x$, the choice $u \propto x + \operatorname{sign}(x_1)\|x\|_2\, e_1$ gives $Hx = -\operatorname{sign}(x_1)\|x\|_2\, e_1$: all entries but the first are zeroed. The sign is chosen to avoid cancellation in $x_1 + \operatorname{sign}(x_1)\|x\|_2$.
- First, we find a Householder matrix $H_1$ such that (we illustrate on a $4 \times 3$ matrix)

$$ H_1 A = \begin{bmatrix} * & * & * \\ 0 & * & * \\ 0 & * & * \\ 0 & * & * \end{bmatrix}. $$

Then,

$$ H_2 H_1 A = \begin{bmatrix} * & * & * \\ 0 & * & * \\ 0 & 0 & * \\ 0 & 0 & * \end{bmatrix}, $$

where

$$ H_2 = \begin{bmatrix} 1 & 0 \\ 0 & H'_2 \end{bmatrix}, $$

and $H'_2$ is a $3 \times 3$ Householder matrix: it does not touch the first column, so the zeros of the first column survive.

Finally,

$$ H_3 H_2 H_1 A = \begin{bmatrix} * & * & * \\ 0 & * & * \\ 0 & 0 & * \\ 0 & 0 & 0 \end{bmatrix} = R, $$

so $Q = (H_3 H_2 H_1)^* = H_1 H_2 H_3$ (each $H_j$ is Hermitian).

It is simple to implement; here is a short version for real matrices. The cost is about $2nm^2 - \frac23 m^3$ operations for $R$.

In [ ]:
def householder_qr(A: np.ndarray):
    # full QR of a real n x m matrix by Householder reflections
    R = np.array(A, dtype=float)
    n, m = R.shape
    Q = np.eye(n)
    for j in range(min(n - 1, m)):
        x = R[j:, j]
        u = x.copy()
        u[0] += np.copysign(np.linalg.norm(x), x[0])    # the sign avoids cancellation
        norm_u = np.linalg.norm(u)
        if norm_u == 0.0:
            continue                                   # the column is already zero
        u /= norm_u
        R[j:, j:] -= 2.0 * np.outer(u, u @ R[j:, j:])  # R := H_j R
        Q[:, j:] -= 2.0 * np.outer(Q[:, j:] @ u, u)    # Q := Q H_j
    return Q, np.triu(R)


A = rng.standard_normal((6, 4))
Q, R = householder_qr(A)
print(f'||A - QR|| = {np.linalg.norm(A - Q @ R):.1e},  ||Q^T Q - I|| = {orth_error(Q):.1e}')


## Four ways to compute QR: which one keeps orthogonality?

We compare classical Gram&ndash;Schmidt (CGS), modified Gram&ndash;Schmidt (MGS), Cholesky QR and Householder reflections on $100 \times 20$ matrices $A = U\Sigma V^\top$ with prescribed condition numbers $\kappa(A)$ from $10$ to $10^{14}$.

**Guess first:** which curve will be flat?

In [ ]:
def cgs(A: np.ndarray):
    n, m = A.shape
    Q, R = np.zeros((n, m)), np.zeros((m, m))
    for k in range(m):
        R[:k, k] = Q[:, :k].T @ A[:, k]              # all projections of the original a_k
        v = A[:, k] - Q[:, :k] @ R[:k, k]
        R[k, k] = np.linalg.norm(v)
        Q[:, k] = v / R[k, k]
    return Q, R


def mgs(A: np.ndarray):
    n, m = A.shape
    V, Q, R = np.array(A, dtype=float), np.zeros((n, m)), np.zeros((m, m))
    for k in range(m):
        R[k, k] = np.linalg.norm(V[:, k])
        Q[:, k] = V[:, k] / R[k, k]
        R[k, k + 1:] = Q[:, k] @ V[:, k + 1:]       # subtract q_k from all remaining vectors at once
        V[:, k + 1:] -= np.outer(Q[:, k], R[k, k + 1:])
    return Q, R


def cholesky_qr(A: np.ndarray):
    L = np.linalg.cholesky(A.T @ A)                  # A^T A = L L^T, R = L^T
    return sla.solve_triangular(L, A.T, lower=True).T, L.T


n, m = 100, 20
U, _ = np.linalg.qr(rng.standard_normal((n, m)))
V, _ = np.linalg.qr(rng.standard_normal((m, m)))
exponents = np.arange(1, 15)
methods = {'CGS': lambda A: cgs(A)[0],
           'MGS': lambda A: mgs(A)[0],
           'Cholesky QR': lambda A: cholesky_qr(A)[0],
           'Householder': lambda A: householder_qr(A)[0][:, :A.shape[1]]}
errors = {name: [] for name in methods}
for p in exponents:
    A = U @ np.diag(np.logspace(0, -p, m)) @ V.T     # kappa(A) = 10^p
    for name, method in methods.items():
        try:
            errors[name].append(orth_error(method(A)))
        except np.linalg.LinAlgError:
            errors[name].append(np.nan)                # Cholesky of the computed Gram matrix failed

kappa = 10.0 ** exponents
fig, ax = plt.subplots()
for name, err in errors.items():
    ax.loglog(kappa, err, 'o-', ms=4, label=name)
ax.loglog(kappa, 1e-16 * kappa, 'k:', lw=1, label=r'$\kappa\,\varepsilon$')
ax.loglog(kappa, 1e-16 * kappa ** 2, 'k--', lw=1, label=r'$\kappa^2\varepsilon$')
ax.set_xlabel(r'$\kappa(A)$'); ax.set_ylabel(r'$\|Q^\top Q - I\|_F$'); ax.set_ylim(1e-17, 1e2)
ax.legend(fontsize=8)
plt.show()


- **Householder**: $\|Q^\top Q - I\| \approx \varepsilon_{\rm mach}$ for any $\kappa(A)$.
- **MGS**: the loss of orthogonality grows roughly like $\kappa(A)\,\varepsilon_{\rm mach}$.
- **CGS** and **Cholesky QR**: roughly like $\kappa(A)^2\varepsilon_{\rm mach}$, until they break down completely (missing points: the Cholesky factorization of the computed Gram matrix failed).

This is why LAPACK and `numpy.linalg.qr` use Householder reflections. MGS is still useful when $Q$ only needs to be approximately orthogonal (e.g. in Krylov methods), and reorthogonalization cures CGS.

## QR decomposition: real life

- In reality, since this is a dense matrix factorization, you should implement the algorithm in terms of **blocks** (why?).

- Several Householder reflections are accumulated into the **compact WY representation**

$$H_1 H_2 \cdots H_b = I - V T V^*,$$

where the $n\times b$ matrix $V$ stores the Householder vectors and $T$ is a $b \times b$ upper triangular matrix. (A product of reflections is not a reflection itself, so it is not of the form $I - 2UU^*$ with $U^*U = I$.)

- Applying $I - VTV^*$ to the rest of the matrix is a sequence of matrix-matrix products, i.e. **BLAS-3** operations: they reuse data in cache and run close to the peak speed of the processor.

- In LAPACK, `geqrf` computes $R$ and stores the reflectors in place of the zeros; `orgqr` forms $Q$ explicitly only if needed. `numpy.linalg.qr` and `scipy.linalg.qr` call these routines.

## Rank-revealing QR decomposition

- The QR decomposition can also be used to compute the (numerical) rank of a matrix, see
[Rank-Revealing QR Factorizations and the Singular Value Decomposition, Y. P. Hong; C.-T. Pan](http://scgroup.hpclab.ceid.upatras.gr/class/SCII/Various/HongPan.pdf).

- It is done via a so-called **rank-revealing** factorization.

- It is based on the representation

$$A P = Q R,$$

where $P$ is a permutation matrix (multiplication from the **right** permutes the columns), and $R$ has the block form

$$R = \begin{bmatrix} R_{11} & R_{12} \\ 0 & R_{22}\end{bmatrix}, \qquad R_{11} \in \mathbb{C}^{r\times r}.$$

- The goal is to find $P$ such that the norm of $R_{22}$ is **small**, so you can find the numerical rank by looking at it.

- An estimate is $\sigma_{r+1}(A) \leq \Vert R_{22} \Vert_2$ (check why).

- The simplest strategy is **column pivoting** (Businger&ndash;Golub): at every step take the remaining column of the largest norm; `scipy.linalg.qr(A, pivoting=True)`.

<details><summary><b>Why</b></summary>

$Q$ has orthonormal columns and $P$ is a permutation, so $A$ and $R$ have the same singular values. The matrix $\begin{bmatrix} R_{11} & R_{12} \\ 0 & 0 \end{bmatrix}$ has rank at most $r$ and differs from $R$ by $\begin{bmatrix} 0 & 0 \\ 0 & R_{22} \end{bmatrix}$. By the Eckart&ndash;Young theorem, $\sigma_{r+1}(R) \le \|R_{22}\|_2$.

</details>

In [ ]:
n, m, rank = 50, 30, 5
A = rng.standard_normal((n, rank)) @ rng.standard_normal((rank, m))   # rank 5
A[:, :10] = np.outer(A[:, 0], rng.standard_normal(10))                # first 10 columns: multiples of one vector
A += 1e-10 * rng.standard_normal((n, m))                                # small noise

_, R_plain = np.linalg.qr(A)
_, R_piv, perm = sla.qr(A, mode='economic', pivoting=True)            # A[:, perm] = Q R
sv = np.linalg.svd(A, compute_uv=False)

fig, ax = plt.subplots()
ax.semilogy(np.abs(np.diag(R_plain)), 'o-', ms=4, label='|diag R|, no pivoting')
ax.semilogy(np.abs(np.diag(R_piv)), 's-', ms=4, label='|diag R|, column pivoting')
ax.semilogy(sv, 'k.', label='singular values')
ax.set_xlabel('index'); ax.legend(fontsize=8)
plt.show()
print(f'sigma_{rank + 1} = {sv[rank]:.1e},  ||R_22||_2 with pivoting = {np.linalg.norm(R_piv[rank:, rank:], 2):.1e}')


Without pivoting, $|r_{kk}|$ drops to the noise level already at $k = 2$ (the first columns are parallel) and then jumps up again: small diagonal entries do not reveal the rank. With column pivoting the diagonal of $R$ follows the singular values, drops after $r = 5$ entries, and $\|R_{22}\|_2$ is of the order of $\sigma_6$.

## Summary of the first part

- LU and QR decompositions can be computed by **direct methods** in a finite number of operations.

- What about the Schur form and the SVD?

- They cannot be computed by **direct methods** (why?): they can only be computed by **iterative methods**.

- In practice, iterative methods still have $\mathcal{O}(n^3)$ total complexity in floating point arithmetic, thanks to fast convergence.

<details><summary><b>Why</b></summary>

Eigenvalues are the roots of the characteristic polynomial, and every polynomial is the characteristic polynomial of its companion matrix. By the Abel&ndash;Ruffini theorem there is no formula in radicals for the roots of a general polynomial of degree $\ge 5$, hence no finite algorithm using arithmetic operations and $k$-th roots computes the eigenvalues. The same holds for singular values: $\sigma_i^2$ are the eigenvalues of $A^*A$.

</details>

## Schur form

- Recall that every matrix can be written in the **Schur form**

$$A = Q T Q^*,$$

with an upper triangular $T$ and a unitary $Q$, and this decomposition gives the eigenvalues of the matrix (they are on the diagonal of $T$).

- The first and the main algorithm for computing the Schur form is the **QR algorithm**.

## QR algorithm

- The QR algorithm was proposed independently by Kublanovskaya (1961) and Francis (1961&ndash;1962).

- <font color='red'> Do not **mix up** the QR algorithm and the QR decomposition! </font>

- The QR decomposition is a representation of a matrix, whereas the QR algorithm uses QR decompositions to compute the eigenvalues!

## Way to QR algorithm

- Consider the equation

$$A = Q T Q^*,$$

and rewrite it in the form

$$ Q T = A Q. $$

- On the left we can see the QR factorization of the matrix $AQ$.

- We can use this to derive a fixed-point iteration for the Schur form, also known as the **QR algorithm**.

## Derivation of the QR algorithm as a fixed-point iteration

To avoid a clash of notation, denote the accumulated unitary matrix by $\overline Q_k$. We iterate the fixed-point equation $A\overline Q = \overline Q T$ as

$$
    \overline Q_k R_k = A \overline Q_{k-1}, \qquad \overline Q_0 = I,
$$

i.e. at every step we compute the QR decomposition of $A\overline Q_{k-1}$. Since $\overline Q_{k-1}$ is unitary, this also gives $\overline Q_k^* A = R_k \overline Q_{k-1}^*$.

Introduce

$$A_k = \overline Q_k^* A \overline Q_k, \qquad Q_k = \overline Q_{k-1}^* \overline Q_k \ \ \text{(unitary)}.$$

Then

$$A_{k-1} = \overline Q_{k-1}^* A \overline Q_{k-1} = \overline Q_{k-1}^* \overline Q_k R_k = Q_k R_k,$$

and the new approximation reads

$$A_k = \overline Q_k^* A \overline Q_k = R_k \overline Q_{k-1}^* \overline Q_k = R_k Q_k.$$

So we arrive at the standard form of the QR algorithm, and $\overline Q_k = \overline Q_{k-1} Q_k = Q_1 Q_2 \cdots Q_k$.

The final formulas are written in the classical **QR&ndash;RQ** form:

1. Start from $A_0 = A$.
2. For $k = 1, 2, \dots$: compute the QR factorization $A_{k-1} = Q_k R_k$.
3. Set $A_k = R_k Q_k$.

Iterate until $A_k$ is **triangular enough** (e.g. the norm of the strictly lower triangular part is small enough).

**Notation.** $A_k$ is the $k$-th iterate and $A^k$ is the $k$-th power of $A$; both appear below, and they are different matrices.

## What about the convergence and complexity

**Statement.** The matrices $A_k$ are unitarily similar to $A$:

$$A_k = R_k Q_k = Q_k^* (Q_k R_k) Q_k = Q_k^* A_{k-1} Q_k = (Q_1 \cdots Q_k)^* A\, (Q_1 \cdots Q_k) = \overline Q_k^* A\, \overline Q_k,$$

and the product of unitary matrices is a unitary matrix. Hence all $A_k$ have the same eigenvalues as $A$.

- The complexity of each step is $\mathcal{O}(n^3)$ if a general QR decomposition is used.

- Our hope is that $A_k$ will be **very close to a triangular matrix** for sufficiently large $k$.

In [ ]:
n = 40
idx = np.arange(n)
a = 1.0 / (idx[None, :] - idx[:, None] + 0.5)        # a[j, i] = 1 / (i - j + 0.5)
ak = a.copy()
for _ in range(20_000):
    q, r = np.linalg.qr(ak)
    ak = r @ q

print('Block A_k[4:8, 4:8]:')
print(np.array2string(ak[4:8, 4:8], precision=3))

ev = np.linalg.eigvals(a)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
im = axes[0].imshow(np.log10(np.abs(ak) + 1e-300), vmin=-16, vmax=1, cmap='viridis')
fig.colorbar(im, ax=axes[0], label=r'$\log_{10}|(A_k)_{ij}|$')
axes[0].set_title('$A_k$ after 20000 QR steps', fontsize=10)
axes[1].plot(ev.real, ev.imag, 'o', ms=4)
axes[1].set_xlabel('Re'); axes[1].set_ylabel('Im'); axes[1].set_aspect('equal')
axes[1].set_title(f'eigenvalues of a: {np.sum(np.abs(ev.imag) < 1e-12)} of {n} are real', fontsize=10)
plt.show()


$A_k$ does not become triangular: there are $2\times2$ blocks on the diagonal. The matrix is real, its eigenvalues come in **complex conjugate pairs** $\lambda, \bar\lambda$ of **equal modulus**, and the real QR algorithm cannot separate them. The limit is the **real Schur form**: a block upper triangular matrix with $1\times1$ and $2\times2$ diagonal blocks, where each $2\times2$ block carries a conjugate pair. The entries below the blocks decay (after $10^5$ steps they were about $10^{-103}$), while the blocks themselves keep rotating.

## Convergence and complexity of the QR algorithm

- If $|\lambda_1| > |\lambda_2| > \dots > |\lambda_n|$ (and a mild nondegeneracy condition holds, see the pause below), then $A_k$ converges to an upper triangular matrix with the eigenvalues on the diagonal **ordered by decreasing modulus**.
- The subdiagonal entry $(A_k)_{i+1, i}$ decays as $|\lambda_{i+1}/\lambda_i|^k$: the convergence is **linear** and can be slow.
- Each step is one QR factorization and one matrix-by-matrix product, i.e. $\mathcal{O}(n^3)$ operations.

**Q**: does it mean $\mathcal{O}(n^4)$ complexity in total?

**A**: fortunately, not.

- First, $A$ is reduced to the **upper Hessenberg** form once ($\mathcal{O}(n^3)$); the QR algorithm preserves this form, and a step costs only $\mathcal{O}(n^2)$ ($\mathcal{O}(n)$ for a symmetric tridiagonal matrix).
- Second, we speed up the QR algorithm by **shifts**: $A_k - \sigma I$ has the same Schur vectors. With good shifts, typically 2&ndash;3 iterations per eigenvalue are enough.
- Altogether this gives $\mathcal{O}(n^3)$. The details are in the next lecture; the idea of shifts is shown at the end of today's part.

## Convergence of QR algorithm

Does the QR algorithm always converge?

Provide an example.

## Counterexample

For the matrix $A = \begin{bmatrix} 0 & 1 \\ 1 & 0 \end{bmatrix}$ we have $A = Q_1 R_1$ with $Q_1 = A$, $R_1 = I$, so $A_1 = R_1 Q_1 = A$, and $A_k = A$ for all $k$ (up to signs, depending on the QR routine).

The reason: the eigenvalues $\pm 1$ have **equal modulus**. We will see in a minute that the QR algorithm contains the power method, and the power method fails here as well: started from $e_1$, it alternates between $e_1$ and $e_2$.

## QR algorithm and the power method

In the previous lecture we considered the **power method**: $A^k v$, normalised, approximates the leading eigenvector. The QR algorithm (implicitly) computes the QR factorization of the power $A^k$:

$$A^k = \overline Q_k \overline R_k, \qquad \overline Q_k = Q_1 Q_2 \cdots Q_k, \quad \overline R_k = R_k \cdots R_2 R_1.$$

$\overline Q_k$ is unitary, and $\overline R_k$ is upper triangular as a product of upper triangular matrices.

**Proof** (induction on $k$; for $k=1$ it is $A = A_0 = Q_1 R_1$). From $A_{k-1} = \overline Q_{k-1}^* A \overline Q_{k-1}$ we get

$$A\overline Q_{k-1} = \overline Q_{k-1} A_{k-1} = \overline Q_{k-1} Q_k R_k = \overline Q_k R_k.$$

Hence

$$A^k = A \cdot A^{k-1} = A\, \overline Q_{k-1} \overline R_{k-1} = \overline Q_k R_k \overline R_{k-1} = \overline Q_k \overline R_k. \qquad \blacksquare$$

### What this means

Compare the first columns of $A^k = \overline Q_k \overline R_k$. Since $\overline R_k$ is upper triangular,

$$A^k e_1 = (\overline R_k)_{11}\, \overline Q_k e_1:$$

- the **first column of $\overline Q_k$ is the power method** started from $e_1$, i.e. $A^k e_1 / \|A^k e_1\|_2$ (up to sign);
- the entry $(A_k)_{11} = q_1^* A q_1$, where $q_1 = \overline Q_k e_1$, is the **Rayleigh quotient** &mdash; exactly the eigenvalue estimate of the power method.

The first $j$ columns give more: $A^k [e_1, \dots, e_j] = \overline Q_k[:, 1{:}j]\, \overline R_k[1{:}j, 1{:}j]$, so the first $j$ columns of $\overline Q_k$ span $A^k \operatorname{span}(e_1, \dots, e_j)$. This is the power method applied to a whole subspace (**subspace**, or **orthogonal, iteration**). If $|\lambda_j| > |\lambda_{j+1}|$, this subspace converges to the invariant subspace of the $j$ largest eigenvalues, and in the basis $\overline Q_k$ this means that the block $A_k[j{+}1{:}n,\ 1{:}j]$ tends to zero.

A picture to keep in mind: $\overline Q_k$ is a camera that follows the directions of $A^k$, and $A_k = \overline Q_k^* A\, \overline Q_k$ is the matrix $A$ seen through this camera.

In [ ]:
n = 5
X = rng.standard_normal((n, n))
lam = np.array([4.0, 2.0, 1.0, 0.5, 0.25])
A = X @ np.diag(lam) @ np.linalg.inv(X)         # non-symmetric, known eigenvalues

K = 25
Ak, Qbar = A.copy(), np.eye(n)                  # QR algorithm
V = np.eye(n)                                   # subspace iteration, V_0 = I
x = np.eye(n)[:, 0]                             # power method, x_0 = e_1
for _ in range(K):
    Q, R = np.linalg.qr(Ak)
    Ak, Qbar = R @ Q, Qbar @ Q
    V, _ = np.linalg.qr(A @ V)
    x = A @ x
    x /= np.linalg.norm(x)

print(f'power method vs first column of Qbar_K: |cos| = {abs(x @ Qbar[:, 0]):.15f}')
print(f'(A_K)_11 = {Ak[0, 0]:.12f},  Rayleigh quotient = {x @ A @ x:.12f},  lambda_1 = {lam[0]}')

Rbar = Qbar.T @ np.linalg.matrix_power(A, K)
print(f'A^K = Qbar Rbar: relative size of the lower part of Rbar = {np.linalg.norm(np.tril(Rbar, -1)) / np.linalg.norm(Rbar):.1e}')
print('V_K^T Qbar_K (subspace iteration vs QR algorithm):')
print(np.round(V.T @ Qbar, 6))


- The first column of $\overline Q_K$ coincides with the power method, and $(A_K)_{11}$ with its Rayleigh quotient.
- $\overline Q_K^\top A^K$ is upper triangular up to rounding: this is $A^K = \overline Q_K \overline R_K$.
- Subspace iteration $V_k = \mathrm{qr}(A V_{k-1})$, $V_0 = I$, produces the same matrices: $V_K^\top \overline Q_K$ is diagonal with $\pm1$ (QR is unique up to the signs of the columns). The QR algorithm is subspace iteration written in the moving basis $\overline Q_k$.

### The other end: inverse iteration

Invert and take the conjugate transpose of $A^k = \overline Q_k \overline R_k$ (assume $A$ is nonsingular):

$$(A^*)^{-k} = \overline Q_k\, \overline R_k^{-*},$$

and $\overline R_k^{-*}$ is **lower** triangular. Compare the **last** columns: $(A^*)^{-k} e_n = (\overline R_k^{-*})_{nn}\, \overline Q_k e_n$.

- The **last column of $\overline Q_k$ is the inverse power method** for $A^*$ started from $e_n$; it converges to the eigenvector of $A^*$ for the eigenvalue of smallest modulus.
- Hence $(A_k)_{nn} \to \lambda_n$, the eigenvalue of smallest modulus, at the rate $|\lambda_n/\lambda_{n-1}|^k$.

So the QR algorithm runs the power method in the top-left corner and inverse iteration in the bottom-right corner **simultaneously**, for free. This is the key to shifts: inverse iteration with a shift close to $\lambda_n$ converges very fast.

In [ ]:
y = np.eye(n)[:, -1]                    # inverse iteration for A^T, y_0 = e_n
for _ in range(K):
    y = np.linalg.solve(A.T, y)
    y /= np.linalg.norm(y)

print(f'inverse iteration vs last column of Qbar_K: |cos| = {abs(y @ Qbar[:, -1]):.15f}')
print(f'(A_K)_nn = {Ak[-1, -1]:.12f},  lambda_n = {lam[-1]}')
print('diag(A_K):', np.round(np.diag(Ak), 6))


## Each subdiagonal entry has its own speed

Take a symmetric matrix with eigenvalues $10, 7, 5, 3, 2, 1$. The theory predicts $|(A_k)_{i+1,i}| \sim |\lambda_{i+1}/\lambda_i|^k$ with the ratios $0.7,\ 0.71,\ 0.6,\ 0.67,\ 0.5$. Let us check.

In [ ]:
lam6 = np.array([10.0, 7.0, 5.0, 3.0, 2.0, 1.0])
Q0, _ = np.linalg.qr(rng.standard_normal((6, 6)))
A6 = Q0 @ np.diag(lam6) @ Q0.T                  # symmetric, known eigenvalues

num_iter = 80
sub = np.empty((num_iter, 5))
Ak = A6.copy()
for k in range(num_iter):
    Q, R = np.linalg.qr(Ak)
    Ak = R @ Q
    sub[k] = np.abs(np.diag(Ak, -1))

ks = np.arange(1, num_iter + 1)
fig, ax = plt.subplots()
for i in range(5):
    rate = lam6[i + 1] / lam6[i]
    line, = ax.semilogy(ks, sub[:, i], label=rf'$|a_{{{i + 2},{i + 1}}}|$, ratio {rate:.2f}')
    ax.semilogy(ks, sub[0, i] * rate ** (ks - 1), '--', color=line.get_color(), lw=1)
ax.set_xlabel('iteration $k$'); ax.set_ylim(1e-17, 10); ax.legend(fontsize=8)
ax.set_title(r'solid: QR algorithm, dashed: $|\lambda_{i+1}/\lambda_i|^k$', fontsize=10)
plt.show()
print('diag(A_k):', np.round(np.diag(Ak), 8))


The slopes of the solid lines match the dashed ones: each pair of neighbouring eigenvalues is separated at its own speed, and the slowest pair ($7$ and $5$, ratio $0.71$) determines how long the unshifted algorithm runs. On the diagonal we get the eigenvalues in decreasing order.

## Pause: an already triangular matrix

Take $A = \begin{bmatrix} 1 & 1 \\ \varepsilon & 3 \end{bmatrix}$. For $\varepsilon = 0$ the matrix is already triangular, so the QR algorithm does nothing: the eigenvalues stay on the diagonal in the order $1, 3$, **not** by decreasing modulus.

**Question.** What happens for $\varepsilon = 10^{-10}$?

In [ ]:
num_iter = 60
fig, ax = plt.subplots()
for eps in [0.0, 1e-10]:
    Ak = np.array([[1.0, 1.0], [eps, 3.0]])
    a11 = []
    for _ in range(num_iter):
        Q, R = np.linalg.qr(Ak)
        Ak = R @ Q
        a11.append(Ak[0, 0])
    ax.plot(np.arange(1, num_iter + 1), a11, 'o-', ms=3, label=rf'$\varepsilon = {eps:g}$')
    swap = next((k + 1 for k, v in enumerate(a11) if v > 2), None)
    print(f'eps = {eps:g}: (A_k)_11 first exceeds 2 at k = {swap}')
ax.set_xlabel('iteration $k$'); ax.set_ylabel('$(A_k)_{11}$'); ax.legend()
plt.show()


<details><summary><b>Answer</b></summary>

For about $\log_3 10^{10} \approx 21$ iterations nothing visible happens, and then the eigenvalues swap: $(A_k)_{11} \to 3$.

The first column of $\overline Q_k$ is the power method started from $e_1$. For $\varepsilon = 0$ the vector $e_1$ is an eigenvector for $\lambda = 1$, so it has **no component** along the eigenvector of $\lambda = 3$, and the power method never finds it. For $\varepsilon = 10^{-10}$ this component is tiny, but it grows by the factor $3/1$ at every step.

This is the nondegeneracy condition of the convergence theorem: the starting vectors $e_1, \dots, e_j$ must have nonzero components along the dominant eigenvectors. For a random matrix it holds with probability one, and rounding errors usually create the missing components anyway. But a long plateau does not mean convergence.

</details>

## A glimpse of shifts

A shifted QR step with a shift $\sigma_k$:

$$A_{k-1} - \sigma_k I = Q_k R_k, \qquad A_k = R_k Q_k + \sigma_k I = Q_k^* A_{k-1} Q_k.$$

The last column of $\overline Q_k$ now does inverse iteration with $(A - \sigma I)^{-1}$, which converges fast if $\sigma \approx \lambda_n$.

- **Rayleigh shift** $\sigma_k = (A_{k-1})_{nn}$: the entry $(A_k)_{n,n-1}$ converges quadratically, for symmetric matrices even cubically.
- **Wilkinson shift**: the eigenvalue of the trailing $2\times2$ block of $A_{k-1}$ closest to $(A_{k-1})_{nn}$. It also cures the counterexample $\begin{bmatrix} 0 & 1 \\ 1 & 0 \end{bmatrix}$, where the Rayleigh shift is $0$ and does not help.

When $(A_k)_{n,n-1}$ becomes negligible, an eigenvalue is found, and the algorithm continues with the leading $(n-1)\times(n-1)$ block (**deflation**).

In [ ]:
def last_subdiagonal(A: np.ndarray, num_iter: int, shift: str) -> np.ndarray:
    # |(A_k)_{n,n-1}| for k = 1..num_iter of the (shifted) QR algorithm
    Ak, n, hist = A.copy(), A.shape[0], []
    for _ in range(num_iter):
        if shift == 'rayleigh':
            sigma = Ak[-1, -1]
        elif shift == 'wilkinson':
            mu = np.linalg.eigvals(Ak[-2:, -2:])
            sigma = mu[np.argmin(np.abs(mu - Ak[-1, -1]))].real
        else:
            sigma = 0.0
        Q, R = np.linalg.qr(Ak - sigma * np.eye(n))
        Ak = R @ Q + sigma * np.eye(n)
        hist.append(abs(Ak[-1, -2]))
    return np.array(hist)


fig, ax = plt.subplots()
for shift, style in [('none', 'o-'), ('rayleigh', 's-')]:
    h = last_subdiagonal(A6, 10, shift)
    ax.semilogy(np.arange(1, 11), np.maximum(h, 1e-300), style, ms=4, label=f'shift: {shift}')
ax.set_xlabel('iteration $k$'); ax.set_ylabel('$|(A_k)_{n,n-1}|$'); ax.set_ylim(1e-18, 10); ax.legend()
plt.show()

swap = np.array([[0.0, 1.0], [1.0, 0.0]])
for shift in ['none', 'rayleigh', 'wilkinson']:
    print(f'{shift:9s}: |(A_k)_21| after 1, 2, 3 steps =', last_subdiagonal(swap, 3, shift))


With the Rayleigh shift the last subdiagonal entry of the symmetric $6\times6$ matrix drops to the rounding level in a few iterations, while without shifts it decreases by the factor $|\lambda_6/\lambda_5| = 0.5$ per step. For the swap matrix only the Wilkinson shift helps: one step, and the matrix is diagonal.

## A few words about the SVD

- Last but not least: the **singular value decomposition** of a matrix,

$$A = U \Sigma V^*.$$

- We can compute it via the eigendecomposition of

$$A^* A = V \Sigma^* \Sigma V^*$$

and/or

$$A A^* = U \Sigma \Sigma^* U^*$$

with the QR algorithm, but it is a **bad idea** (cf. Gram matrices): the condition number is squared, and singular values below about $\sqrt{\varepsilon_{\rm mach}}\,\sigma_1 \approx 10^{-8}\sigma_1$ are lost.

- We will discuss methods for computing the SVD later.

In [ ]:
U, _ = np.linalg.qr(rng.standard_normal((6, 3)))
V, _ = np.linalg.qr(rng.standard_normal((3, 3)))
s_true = np.array([1.0, 1e-5, 1e-10])
A = U @ np.diag(s_true) @ V.T

s_svd = np.linalg.svd(A, compute_uv=False)
s_gram = np.sqrt(np.abs(np.linalg.eigvalsh(A.T @ A)))[::-1]   # eigvalsh: ascending order
for name, s in [('svd(A)', s_svd), ('sqrt(eig(A^T A))', s_gram)]:
    rel = np.abs(s - s_true) / s_true
    print(f'{name:17s}', '  '.join(f'{v:.6e}' for v in s), '   rel. errors:', '  '.join(f'{e:.0e}' for e in rel))


$\sigma_3 = 10^{-10}$ is computed accurately by the SVD, but via $A^\top A$ it is lost: $\sigma_3^2 = 10^{-20}$ is far below the rounding errors, about $10^{-16}$, of the entries of $A^\top A$.

## Bonus: CS (Cosine-Sine) decomposition

- Let $Q$ be a square unitary matrix with an even number of rows, split into four blocks of equal size

$$ Q = \begin{bmatrix} Q_{11} & Q_{12} \\ Q_{21} & Q_{22} \end{bmatrix}. $$

- Then there exist unitary matrices $U_1, U_2, V_1, V_2$ such that

$$ \begin{bmatrix} U_1^* & 0 \\ 0 & U_2^* \end{bmatrix} \begin{bmatrix} Q_{11} & Q_{12} \\ Q_{21} & Q_{22} \end{bmatrix} \begin{bmatrix} V_1 & 0 \\ 0 & V_2 \end{bmatrix} = \begin{bmatrix} C & S \\ -S & C \end{bmatrix}, $$

where $C = \mathrm{diag}(c)$ and $S = \mathrm{diag}(s)$ with $c_i \geq 0$, $s_i \geq 0$ and $c_i^2 + s_i^2 = 1$, i.e. $c_i = \cos\theta_i$, $s_i = \sin\theta_i$.

- **Q**: how many SVDs do we have inside the CS decomposition?

- The case of a rectangular matrix with orthonormal columns:

$$ \begin{bmatrix} U_1^* & 0 \\ 0 & U_2^* \end{bmatrix} \begin{bmatrix} Q_{1} \\ Q_{2} \end{bmatrix} V = \begin{bmatrix} C \\ S \end{bmatrix}. $$

- An algorithm for computing this decomposition is presented [here](https://arxiv.org/pdf/0707.1838.pdf); in SciPy it is `scipy.linalg.cossin`.

- This decomposition naturally arises in the problem of finding distances and angles between subspaces: in the rectangular case, $c_i$ are the cosines of the **principal angles** between the column space of $\begin{bmatrix} Q_1 \\ Q_2 \end{bmatrix}$ and the coordinate subspace of the first block of rows (see also `scipy.linalg.subspace_angles`).

In [ ]:
n = 6
p = n // 2
Qf, _ = np.linalg.qr(rng.standard_normal((n, n)))
u, cs, vdh = sla.cossin(Qf, p=p, q=p)            # Qf = u @ cs @ vdh
print(f'||Q - U (CS) V^*|| = {np.linalg.norm(u @ cs @ vdh - Qf):.1e}')
print('middle factor (CS):')
print(np.round(cs, 3))
print('cosines      :', np.round(np.sort(np.abs(np.diag(cs[:p, :p]))), 6))
print('svd of Q_11  :', np.round(np.sort(sla.svdvals(Qf[:p, :p])), 6))
print('sines        :', np.round(np.sort(np.abs(np.diag(cs[:p, p:]))), 6))
print('svd of Q_12  :', np.round(np.sort(sla.svdvals(Qf[:p, p:])), 6))


<details><summary><b>Answer</b></summary>

Four, with shared singular vectors. Comparing the blocks,

$$Q_{11} = U_1 C V_1^*, \quad Q_{12} = U_1 S V_2^*, \quad Q_{21} = -U_2 S V_1^*, \quad Q_{22} = U_2 C V_2^*$$

(the third one is an SVD up to the sign of a unitary factor). The singular values of the diagonal blocks are the cosines, those of the off-diagonal blocks are the sines. In the rectangular case $Q_1 = U_1 C V^*$ and $Q_2 = U_2 S V^*$: two SVDs with the same right singular vectors. The signs in the middle factor depend on the convention; SciPy may place the minus sign in the other off-diagonal block.

</details>

## Summary

- QR decomposition: Gram&ndash;Schmidt (loses orthogonality), modified Gram&ndash;Schmidt (better), Householder reflections (orthogonal to machine precision); never via the Gram matrix.
- Rank-revealing QR with column pivoting estimates the numerical rank.
- Schur decomposition and the QR algorithm: $A_k = \overline Q_k^* A\, \overline Q_k$.
- The QR algorithm is the power method in disguise: $A^k = \overline Q_k \overline R_k$; the first columns of $\overline Q_k$ do subspace iteration, the last column does inverse iteration.
- Hence the convergence conditions (eigenvalues of different moduli) and the cure: shifts.

## Next steps

- Efficient implementation of the QR algorithm and its convergence
- Efficient computation of the SVD: 4 algorithms
- More applications of the SVD

### Questions?